# U05_C1 — Dot Plots and Global Alignment
**Course:** Concepts of Bioinformatics (CBBI-501) · **Unit 5:** Sequence Alignment and Database Searching
**Type:** Course notebook · **Time:** 40 min · **Difficulty:** ★★☆
**Goes with:** Lectures L10–L11, Practical S6 (BLAST)
**Before this:** U04_C2

## 1. What you will learn
- **Construct** and interpret 2D dot plots to visualize sequence identity, insertions, and tandem repeats [LO3].
- **Explain** dynamic programming and the Needleman–Wunsch global alignment recurrence relation [LO3].
- **Fill** a global alignment scoring matrix and perform traceback by hand and in Python [LO3].
- **Compare** normal hemoglobin A (HbA) against sickle-cell hemoglobin (HbS) globally [LO3].


## 2. Biological question
Sickle-cell disease is caused by a single point mutation in human *HBB* substituting valine for glutamic acid at codon 6 (E6V). In globin evolution, how do we systematically compare two protein sequences across their entire length to align corresponding functional residues, score identical and mismatched amino acids, and prove that two sequences are homologous?


In [ ]:
import sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "biopython"], check=True)

NOTEBOOK_ID = "U05_C1"
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").is_dir())
DATA_DIR = ROOT / "data"
OUT_DIR = ROOT / "outputs" / NOTEBOOK_ID
OUT_DIR.mkdir(parents=True, exist_ok=True)

import Bio
print("Python", sys.version.split()[0], "| Biopython", Bio.__version__)

## 3. Concept 1: Visualizing Alignments with Dot Plots
### Theory
A **dot plot** is a 2D matrix where Sequence 1 is plotted along the x-axis and Sequence 2 along the y-axis. A dot is placed at $(i, j)$ whenever sequence character $x_i$ equals $y_j$.
- **Continuous diagonal**: Regions of uninterrupted sequence identity.
- **Shifted diagonal**: An insertion or deletion (indel) introduced a coordinate offset.
- **Parallel diagonal lines**: Tandem or interspersed sequence repeats.
- **Noise reduction**: A sliding window of size $W$ with match threshold $T$ filters random single-character background noise.


In [ ]:
# Worked Example: Generating a Dot Plot in Python
import numpy as np
import matplotlib.pyplot as plt

seq1 = "VHLTPEEKSAVTALWGKVNV"  # Normal HbA N-terminus
seq2 = "VHLTPVEKSAVTALWGKVNV"  # Sickle-cell HbS N-terminus (E6V)

# Create boolean identity matrix
dot_matrix = np.zeros((len(seq2), len(seq1)), dtype=int)
for i in range(len(seq2)):
    for j in range(len(seq1)):
        if seq2[i] == seq1[j]:
            dot_matrix[i, j] = 1

fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(dot_matrix, cmap="binary", origin="upper")
ax.set_xticks(range(len(seq1)))
ax.set_xticklabels(list(seq1))
ax.set_yticks(range(len(seq2)))
ax.set_yticklabels(list(seq2))
ax.set_xlabel("Seq 1: HbA (Normal)")
ax.set_ylabel("Seq 2: HbS (Sickle-cell)")
ax.set_title("Dot Plot: HbA vs HbS N-terminus")
plt.tight_layout()
plt.savefig(OUT_DIR / "dotplot_hba_hbs.png", dpi=100)
plt.show()


## 4. Concept 2: Needleman–Wunsch Global Alignment
### Theory
Needleman and Wunsch (1970) introduced dynamic programming to guarantee finding the mathematically optimal global alignment across the entire length of two sequences.

Given sequences $X$ of length $M$ and $Y$ of length $N$:
1. **Matrix Initialization**:
   $$F(0, 0) = 0$$
   $$F(i, 0) = i \times d \quad (i = 1 \dots M)$$
   $$F(0, j) = j \times d \quad (j = 1 \dots N)$$
   where $d$ is the linear gap penalty (negative score).

2. **Recurrence Relation**:
   $$F(i, j) = \max \begin{cases}
   F(i-1, j-1) + s(x_i, y_j) & \text{(Diagonal: match/mismatch)} \\
   F(i-1, j) + d & \text{(Up: gap in sequence } Y) \\
   F(i, j-1) + d & \text{(Left: gap in sequence } X)
   \end{cases}$$

3. **Traceback**: Begins at bottom-right cell $(M, N)$ and traces path pointers back to $(0, 0)$.


In [ ]:
# Worked Example: Hand-sized Needleman-Wunsch Matrix in Python
def needleman_wunsch(seq_x, seq_y, match_score=2, mismatch_score=-1, gap_penalty=-2):
    m, n = len(seq_x), len(seq_y)
    F = np.zeros((m + 1, n + 1), dtype=int)
    
    # Initialize margins
    for i in range(1, m + 1):
        F[i, 0] = i * gap_penalty
    for j in range(1, n + 1):
        F[0, j] = j * gap_penalty
        
    # Fill DP matrix
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            s = match_score if seq_x[i-1] == seq_y[j-1] else mismatch_score
            diag = F[i-1, j-1] + s
            up = F[i-1, j] + gap_penalty
            left = F[i, j-1] + gap_penalty
            F[i, j] = max(diag, up, left)
            
    return F

s_x = "HEAGAWG"
s_y = "PAWHEAE"
dp_matrix = needleman_wunsch(s_x, s_y)

print(f"Aligning {s_x} vs {s_y}")
print("Final Global Alignment Optimal Score F(M, N):", dp_matrix[-1, -1])
print("\nDynamic Programming Matrix:\n", dp_matrix)


In [ ]:
# Try it: Align the first 6 residues of HbA ('VHLTPE') and HbS ('VHLTPV')
hba_sub = "VHLTPE"
hbs_sub = "VHLTPV"
dp_hb = needleman_wunsch(hba_sub, hbs_sub, match_score=2, mismatch_score=-1, gap_penalty=-2)
print("HbA vs HbS (6-mer) optimal global score:", dp_hb[-1, -1])


## 5. Compare with Biopython PairwiseAligner
Biopython provides high-performance pairwise alignment using the `Bio.Align.PairwiseAligner` class.


In [ ]:
# Align using Biopython PairwiseAligner
from Bio.Align import PairwiseAligner

aligner = PairwiseAligner()
aligner.mode = "global"
aligner.match_score = 2
aligner.mismatch_score = -1
aligner.open_gap_score = -2
aligner.extend_gap_score = -2

score = aligner.score(hba_sub, hbs_sub)
print("Biopython computed score:", score)
print("Matches our manual DP matrix score?", score == dp_hb[-1, -1])

alignments = aligner.align(hba_sub, hbs_sub)
print("\nOptimal Global Alignment:")
print(alignments[0])


## 6. Summary
- Dot plots provide an immediate visual representation of sequence relationships and structural rearrangements.
- Needleman–Wunsch guarantees the optimal global alignment over full sequence lengths.
- Initialization fills boundaries with cumulative gap penalties; traceback starts at bottom-right $(M, N)$.


## 7. Next Steps
- Learn about local alignment and substitution matrices in `U05_C2_Local_Alignment_and_Scoring.ipynb`.
- Explore database search heuristics in `U05_C3_Reading_BLAST_Results.ipynb`.
